# pvl-py quickstart

End-to-end walkthrough of the Python SDK for the Peptide Visual Lab. Five cells:

1. Install
2. Analyze a FASTA file → peptide DataFrame
3. Inspect the top rows
4. Rank by Peleg's multi-signal formula
5. Find similar peptides to the top hit

**Prerequisite**: a PVL backend reachable at `http://localhost:8000`. Run `make backend` in the repo root, or set `PVL_SERVER_URL` to your deploy URL.

In [ ]:
# 1. Install pvl-py (editable install from the repo for now;
#    `pip install pvl-py` once it ships on PyPI).
%pip install --quiet -e ../

In [ ]:
# 2. Analyze the shipped 5-peptide FASTA fixture.
#    Same fixture the backend's §H route tests use, so the round-trip is
#    end-to-end verified.
import pvl

df = pvl.analyze("../../backend/tests/fixtures/example.fasta")
print(f"Analyzed {len(df)} peptides")
df.columns.tolist()

In [ ]:
# 3. Spot-check the canonical PVL columns.
df[["id", "sequence", "length", "muH", "hydrophobicity", "ffHelixFlag"]].head()

In [ ]:
# 4. Rank using the 'helix' preset (Peleg-FIX-024).
#    Same formula the web UI ships — see ui/src/lib/ranking.ts.
ranked = pvl.rank(df, preset="helix", top_n=5)
ranked[["id", "score", "reasons"]].head()

In [ ]:
# 5. Find the 5 peptides most similar to the top-ranked one.
#    Similarity is cosine distance on ESM-2 8M embeddings (ADR-017).
top_id = ranked.iloc[0]["id"]
similar = pvl.find_similar(top_id, k=5)
similar[["id", "sequence", "distance"]]

## Next steps

- **Custom weights**: `pvl.rank(df, weights={'muH': 60, 'hydrophobicity': 40}, top_n=10)` — overrides the preset.
- **UniProt search**: `pvl.search_uniprot('amyloid', organism=1280, length_min=10, length_max=50)` returns a DataFrame you can pipe straight into `pvl.analyze(...)`.
- **Async**: every function has an `a*` variant (`await pvl.aanalyze(...)`) for concurrent notebook cells.
- **Different backend**: `pvl.set_default_server('https://pvl.your-lab.edu')` once at the top of the notebook, or set `PVL_SERVER_URL` in the kernel env.

Full API reference: [`pvl-py/README.md`](../README.md).